# Taylor-Green vortex turbulence simulation in Julia

**작성자**: Donghui Son  
**날짜**: 2025.08.14

---

## 소개

본 포스트는 *[3D Pseudo-Spectral Navier-Stokes Solver in Julia](https://youtu.be/QNJeWgVLML8?si=TSyqFho8WjGMtxki)* 영상을 바탕으로 정리한 내용입니다. 3차원 비압축성 Navier-Stokes 방정식을 의사-스펙트럴(pseudo-spectral) 방법으로 해결하여 Taylor-Green vortex의 진화를 시뮬레이션합니다. 이는 난류 연구의 표준 벤치마크로, 비교적 단순하지만 난류로 전이되는 전형적인 과정을 잘 보여줍니다.

### 시뮬레이션의 목표

- 3차원 비압축성 Navier-Stokes 방정식의 수치해
- 층류에서 난류로의 전이 과정 관찰
- 의사-스펙트럴(Pseudo-spectral) 방법의 구현

## 구현 세부사항

### 라이브러리 임포트 (줄 152-155)

In [ ]:
using FFTW            # Fast Fourier Transform 라이브러리
using GLMakie         # GPU 가속 3D 시각화
using Statistics      # 통계 함수 (mean 등)
using ProgressMeter   # 진행 상황 표시

**수학적 의미:**
- `FFTW`: $\mathcal{O}(N^3 \log N)$ 복잡도로 푸리에 변환
- `GLMakie`: 와류장의 실시간 3D 렌더링
- `Statistics`: 수치 안정성을 위한 통계 연산
- `ProgressMeter`: 장시간 시뮬레이션의 진행 상황 추적

### 전역 상수 정의 (줄 157-161)

In [ ]:
N_POINTS_P_AXIS = 50           # 각 축당 격자점 수
KINEMATIC_VISCOSITY = 1.0 / 1_600  # 동점성 계수 ν
TIME_STEP_LENGTH = 0.02        # 시간 간격 Δt
N_TIME_STEPS = 1_700           # 총 시간 단계 수
PLOT_EVERY = 3                 # 시각화 주기

**수학적 의미:**

1. **격자점 수 (N = 50)**:
   - 최대 해상 가능한 파수:
  
    $$
    k_{max} = N/2 = 25
    $$

   - Nyquist 주파수로 제한됨
   - 총 자유도:
  
    $$
    50^3 = 125,000
    $$

2. **동점성 계수 (ν = 1/1600)**:
   - Reynolds 수:
  
    $$
    Re = \frac{UL}{\nu} = \frac{1 \cdot 2\pi}{1/1600} = 3200\pi \approx 10,053
    $$
   
   - $U = 1$: 초기 최대 속도 (특성 속도)
   - $L = 2\pi$: 도메인 크기 (특성 길이)
   - 높은 Re로 난류 발달 가능
   - Kolmogorov 스케일: 
  
    $$
    \eta = (\nu^3/\epsilon)^{1/4}
    $$

3. **시간 간격 (Δt = 0.02)**:
   - CFL 조건: 
  
    $$
    CFL = \frac{u_{max} \Delta t}{\Delta x} < 1
    $$
   
   - 점성 안정성: 
  
    $$
    \Delta t < \frac{\Delta x^2}{4\nu}
    $$

   - 명시적 오일러의 안정성 제약

4. **총 시뮬레이션 시간**:
   - $T_{total} = 1700 \times 0.02 = 34$ time units
   - 약 $5.4 \tau_{eddy}$ ($\tau_{eddy} = 2\pi$ $\rightarrow$ vortex turnover time)

### Cross Product 함수

In [ ]:
function cross_product!(
    res_x::Array{T, 3},
    res_y::Array{T, 3},
    res_z::Array{T, 3},
    a_x::Array{T, 3},
    a_y::Array{T, 3},
    a_z::Array{T, 3},
    b_x::Array{T, 3},
    b_y::Array{T, 3},
    b_z::Array{T, 3},
) where T<:Union{Float64, Complex{Float64}}
    res_x .= (
        a_y .* b_z
        .-
        a_z .* b_y
    )

    res_y .= (
        a_z .* b_x
        .-
        a_x .* b_z
    )

    res_z .= (
        a_x .* b_y
        .-
        a_y .* b_x
    )
end

**함수 시그니처:**
- `!` 표시: In-place 연산 (메모리 효율적)
- 제네릭 타입 `T`: 실수와 복소수 모두 지원
- 3차원 배열: 공간 격자점에서의 벡터장

**수학적 정의:**
$$\mathbf{a} \times \mathbf{b} = \begin{vmatrix}
\mathbf{i} & \mathbf{j} & \mathbf{k} \\
a_x & a_y & a_z \\
b_x & b_y & b_z
\end{vmatrix} = (a_y b_z - a_z b_y)\mathbf{i} + (a_z b_x - a_x b_z)\mathbf{j} + (a_x b_y - a_y b_x)\mathbf{k}
$$

**사용 목적:**
1. **와도 계산**: $\boldsymbol{\omega} = \nabla \times \mathbf{u} = i\mathbf{k} \times \hat{\mathbf{u}}$ (푸리에 공간)
2. **대류항 계산**: $\mathbf{m} = \mathbf{u} \times \boldsymbol{\omega}$ (물리 공간)


### Main 함수 

In [ ]:
function main()

#### 공간 격자 생성 (줄 194-200)

In [ ]:
# 줄 194-196: 1차원 좌표 범위 생성
x_range = range(0.0, 2*pi, N_POINTS_P_AXIS+1)[1:end-1]
y_range = range(0.0, 2*pi, N_POINTS_P_AXIS+1)[1:end-1]
z_range = range(0.0, 2*pi, N_POINTS_P_AXIS+1)[1:end-1]

**수학적 설명:**
- 도메인: $\Omega = [0, 2\pi]^3$
- 격자 간격: $\Delta x = \Delta y = \Delta z = \frac{2\pi}{N} = \frac{2\pi}{50}$
- `[1:end-1]`: 주기 경계조건으로 마지막 점 제외
  - $x_0 = 0$과 $x_N = 2\pi$는 동일한 점

In [ ]:
# 줄 198-200: 3차원 좌표 격자 생성
coordinates_x = [x for x in x_range, y in y_range, z in z_range]
coordinates_y = [y for x in x_range, y in y_range, z in z_range]
coordinates_z = [z for x in x_range, y in y_range, z in z_range]

**데이터 구조:**
- 각 배열: `50×50×50` 크기
- `coordinates_x[i,j,k]`: 격자점 `(i,j,k)`의 x 좌표
- 메모리 레이아웃: Column-major (Julia 기본)

#### 파수 공간 설정 (줄 202-216)

In [ ]:
# 줄 202-204: 1차원 파수 생성
wavenumbers_1d_x = fftfreq(N_POINTS_P_AXIS) .* N_POINTS_P_AXIS
wavenumbers_1d_y = fftfreq(N_POINTS_P_AXIS) .* N_POINTS_P_AXIS
wavenumbers_1d_z = fftfreq(N_POINTS_P_AXIS) .* N_POINTS_P_AXIS

**파수 분포:**
- `fftfreq(N)`:
  
  $$
  [0, 1/N, 2/N, ..., (N/2-1)/N, -N/2/N, ..., -1/N]
  $$

- 스케일링 후:

  $$
  k \in [0, 1, 2, ..., N/2-1, -N/2, ..., -1]
  $$

- Nyquist 주파수:
  
  $$
  k_{\text{Nyquist}} = N/2 = 25
  $$


In [ ]:
# 줄 206-208: 3차원 파수 격자
wavenumbers_x = [k_x for k_x in wavenumbers_1d_x, k_y in wavenumbers_1d_y, k_z in wavenumbers_1d_z]
wavenumbers_y = [k_y for k_x in wavenumbers_1d_x, k_y in wavenumbers_1d_y, k_z in wavenumbers_1d_z]
wavenumbers_z = [k_z for k_x in wavenumbers_1d_x, k_y in wavenumbers_1d_y, k_z in wavenumbers_1d_z]

**파수 벡터:**

$$
\mathbf{k} = (k_x, k_y, k_z)
$$


In [ ]:
# 줄 210-216: 파수 크기 계산
wavenumbers_norm = sqrt.(
    wavenumbers_x.^2 + wavenumbers_y.^2 + wavenumbers_z.^2
)

**파수 크기:**

$$
|\mathbf{k}| = \sqrt{k_x^2 + k_y^2 + k_z^2}
$$

#### 정규화된 파수 (줄 218-221)

In [ ]:
wavenumbers_norm[iszero.(wavenumbers_norm)] .= 1.0  # k=0 처리
normalized_wavenumbers_x = wavenumbers_x ./ wavenumbers_norm
normalized_wavenumbers_y = wavenumbers_y ./ wavenumbers_norm
normalized_wavenumbers_z = wavenumbers_z ./ wavenumbers_norm

**수학적 의미:**

$$
\hat{\mathbf{k}} = \frac{\mathbf{k}}{|\mathbf{k}|} = \left(\frac{k_x}{|\mathbf{k}|}, \frac{k_y}{|\mathbf{k}|}, \frac{k_z}{|\mathbf{k}|}\right)
$$

**용도:**
- 압력 투영:
  
  $$
  \mathbf{u}_{div-free} = \mathbf{u} - \hat{\mathbf{k}}(\hat{\mathbf{k}} \cdot \mathbf{u})
  $$

- $k=0$ 모드: DC(Direct Current) 성분 (평균값)

#### Taylor-Green 초기조건 (줄 223-226)

In [ ]:
velocity_x = sin.(coordinates_x) .* cos.(coordinates_y) .* cos.(coordinates_z)
velocity_y = -cos.(coordinates_x) .* sin.(coordinates_y) .* cos.(coordinates_z)
velocity_z = zeros((N_POINTS_P_AXIS, N_POINTS_P_AXIS, N_POINTS_P_AXIS))

**수학적 형태:**

$$
u(x,y,z,0) = \sin(x)\cos(y)\cos(z)
$$

$$
v(x,y,z,0) = -\cos(x)\sin(y)\cos(z)
$$

$$
w(x,y,z,0) = 0
$$

**특성 검증:**

1. **비압축성 조건**:
   
$$
\nabla \cdot \mathbf{u} = \frac{\partial u}{\partial x} + \frac{\partial v}{\partial y} + \frac{\partial w}{\partial z} = \cos(x)\cos(y)\cos(z) - \cos(x)\cos(y)\cos(z) + 0 = 0
$$

2. **초기 운동 에너지**:
   
$$
E_0 = \frac{1}{2}\int_\Omega (u^2 + v^2 + w^2) d\mathbf{x} = \frac{\pi^3}{4}
$$

적분 계산:
- $\int_0^{2\pi} \sin^2(x)dx = \int_0^{2\pi} \cos^2(x)dx = \pi$
- $E_0 = \frac{1}{2} \cdot 2 \cdot \pi \cdot \pi \cdot \pi \cdot \frac{1}{4} = \frac{\pi^3}{4}$


#### FFT 계획 생성 (줄 229-231)

In [ ]:
fft_operator = plan_fft(velocity_x; flags=FFTW.MEASURE)
# 줄 231: velocity_x 재초기화 (FFTW.MEASURE가 입력 배열을 변경할 수 있음)
velocity_x = sin.(coordinates_x) .* cos.(coordinates_y) .* cos.(coordinates_z)

**FFTW 계획:**
- `FFTW.MEASURE`: 여러 알고리즘 테스트 후 최적 선택
- **주의**: FFTW.MEASURE는 최적 알고리즘을 찾기 위해 입력 배열을 변경할 수 있음
- 따라서 줄 231에서 velocity_x를 다시 초기화해야 함
- 계획 재사용으로 성능 향상 (약 20-30%)
- In-place/out-of-place 변환 지원

#### 푸리에 공간 초기화 (줄 234-236)

In [ ]:
velocity_x_fft = fft_operator * velocity_x
velocity_y_fft = fft_operator * velocity_y
velocity_z_fft = fft_operator * velocity_z

**푸리에 변환 (FFTW 정규화):**
- 순방향 FFT:
  
  $$
  \hat{u}(\mathbf{k}) = \mathcal{F}[u(\mathbf{x})] = \sum_{\mathbf{x}} u(\mathbf{x}) e^{-i\mathbf{k} \cdot \mathbf{x}}
  $$

- 역방향 FFT: 
  
  $$
  u(\mathbf{x}) = \mathcal{F}^{-1}[\hat{u}(\mathbf{k})] = \frac{1}{N^3}\sum_{\mathbf{k}} \hat{u}(\mathbf{k}) e^{i\mathbf{k} \cdot \mathbf{x}}
  $$

**주의**: FFTW는 순방향 변환에서 정규화하지 않고, 역변환에서 $1/N^3$ 팩터를 적용합니다.

#### 메모리 사전 할당 (줄 233-258)

In [ ]:
# 물리 공간 와도
curl_x = zeros(Float64, (N_POINTS_P_AXIS, N_POINTS_P_AXIS, N_POINTS_P_AXIS))
curl_y = zeros(Float64, (N_POINTS_P_AXIS, N_POINTS_P_AXIS, N_POINTS_P_AXIS))
curl_z = zeros(Float64, (N_POINTS_P_AXIS, N_POINTS_P_AXIS, N_POINTS_P_AXIS))

# 와도 제곱 (시각화용)
curl_x_squared = zeros(Float64, (N_POINTS_P_AXIS, N_POINTS_P_AXIS, N_POINTS_P_AXIS))
curl_y_squared = zeros(Float64, (N_POINTS_P_AXIS, N_POINTS_P_AXIS, N_POINTS_P_AXIS))
curl_z_squared = zeros(Float64, (N_POINTS_P_AXIS, N_POINTS_P_AXIS, N_POINTS_P_AXIS))

# 와도 크기
curl_magnitude = zeros(Float64, (N_POINTS_P_AXIS, N_POINTS_P_AXIS, N_POINTS_P_AXIS))

# 푸리에 공간 와도
curl_x_fft = zeros(Complex{Float64}, (N_POINTS_P_AXIS, N_POINTS_P_AXIS, N_POINTS_P_AXIS))
curl_y_fft = zeros(Complex{Float64}, (N_POINTS_P_AXIS, N_POINTS_P_AXIS, N_POINTS_P_AXIS))
curl_z_fft = zeros(Complex{Float64}, (N_POINTS_P_AXIS, N_POINTS_P_AXIS, N_POINTS_P_AXIS))

# 대류항 (물리 공간)
convection_x = zeros(Float64, (N_POINTS_P_AXIS, N_POINTS_P_AXIS, N_POINTS_P_AXIS))
convection_y = zeros(Float64, (N_POINTS_P_AXIS, N_POINTS_P_AXIS, N_POINTS_P_AXIS))
convection_z = zeros(Float64, (N_POINTS_P_AXIS, N_POINTS_P_AXIS, N_POINTS_P_AXIS))

# 대류항 (푸리에 공간)
convection_x_fft = zeros(Complex{Float64}, (N_POINTS_P_AXIS, N_POINTS_P_AXIS, N_POINTS_P_AXIS))
convection_y_fft = zeros(Complex{Float64}, (N_POINTS_P_AXIS, N_POINTS_P_AXIS, N_POINTS_P_AXIS))
convection_z_fft = zeros(Complex{Float64}, (N_POINTS_P_AXIS, N_POINTS_P_AXIS, N_POINTS_P_AXIS))

**메모리 계산:**
- 실수 배열: $50^3 \times 8$ bytes = 1 MB 각각
- 복소수 배열: $50^3 \times 16$ bytes = 2 MB 각각
- 총 메모리: 약 20 MB

#### De-Aliasing: 구현 (줄 260-268)

In [ ]:
k_max_dealias = 2.0/3.0 * (N_POINTS_P_AXIS//2 + 1)
dealias = (
    (abs.(wavenumbers_x) .< k_max_dealias)
    .*
    (abs.(wavenumbers_y) .< k_max_dealias)
    .*
    (abs.(wavenumbers_z) .< k_max_dealias)
)

1. **앨리어싱(aliasing) 문제**:
   - 비선형항 $u \cdot u$의 푸리에 변환은 convolution
   - Convolution은 파수를 더함: $k_1 + k_2$
   - $k_1 + k_2 > k_{Nyquist}$면 앨리어싱 발생

2. **2/3 규칙**:
   - 최대 유지 파수:
  
   $$
   k_{max} = \frac{2}{3} \times (\frac{N}{2} + 1)
   $$

   - $N=50$ 격자: 
  
   $$
   k_{max} = \frac{2}{3} \times 26 \approx 17.33
   $$

   - 파수 필터링:
  
   $$
   |k_x|, |k_y|, |k_z| < 17.33
   $$

   - $17.33 + 17.33 = 34.66 < 50$ 이므로 앨리어싱 방지

3. **필터 함수**:
   
$$
H(\mathbf{k}) = \begin{cases}
1 & \text{if } |k_x|, |k_y|, |k_z| < k_{max} \\
0 & \text{otherwise}
\end{cases}
$$

#### Time evolution 루프 (줄 291)

In [ ]:
@showprogress "Simulating & Animating ..." for t in 1:N_TIME_STEPS

**시간 적분:**
- 총 1700 단계
- 각 단계: Δt = 0.02
- 총 시간: T = 34

##### 단계 1: 푸리에 공간에서 와도 계산 (줄 292-303)

In [ ]:
cross_product!(
    curl_x_fft, curl_y_fft, curl_z_fft,
    im .* wavenumbers_x,    # ik_x
    im .* wavenumbers_y,    # ik_y
    im .* wavenumbers_z,    # ik_z
    velocity_x_fft,         # û
    velocity_y_fft,         # v̂
    velocity_z_fft,         # ŵ
)

**수학적 연산:**

$$
\hat{\boldsymbol{\omega}} = \mathcal{F}[\nabla \times \mathbf{u}] = i\mathbf{k} \times \hat{\mathbf{u}}
$$

**성분별 계산:**

$$
\hat{\omega}_x = i(k_y \hat{w} - k_z \hat{v})
$$

$$
\hat{\omega}_y = i(k_z \hat{u} - k_x \hat{w})
$$

$$
\hat{\omega}_z = i(k_x \hat{v} - k_y \hat{u})
$$

##### 단계 2: 와도를 물리 공간으로 변환 (줄 305-308)

In [ ]:
curl_x .= real(fft_operator \ curl_x_fft)
curl_y .= real(fft_operator \ curl_y_fft)
curl_z .= real(fft_operator \ curl_z_fft)

**역 푸리에 변환:**

$$
\boldsymbol{\omega}(\mathbf{x}) = \mathcal{F}^{-1}[\hat{\boldsymbol{\omega}}(\mathbf{k})] = \sum_{\mathbf{k}} \hat{\boldsymbol{\omega}}(\mathbf{k}) e^{i\mathbf{k} \cdot \mathbf{x}}
$$

**`real()` 함수:**
- 수치 오차로 인한 작은 허수부 제거
- 실수 입력 → FFT → IFFT → 실수 출력

##### 단계 3: 대류항 계산 (줄 310-321)

In [ ]:
cross_product!(
    convection_x, convection_y, convection_z,
    velocity_x, velocity_y, velocity_z,
    curl_x, curl_y, curl_z,
)

**수학적 의미:**

$$
\mathbf{m} = \mathbf{u} \times \boldsymbol{\omega}
$$

**물리적 해석:**
- Lamb vector라고도 불림
- 비선형 대류항의 대체 표현
  
  $$
  (\mathbf{u} \cdot \nabla)\mathbf{u} = -\mathbf{u} \times \boldsymbol{\omega} + \nabla(|\mathbf{u}|^2/2)
  $$


##### 단계 4: 대류항을 푸리에 공간으로 (줄 323-326)

In [ ]:
convection_x_fft .= fft_operator * convection_x
convection_y_fft .= fft_operator * convection_y
convection_z_fft .= fft_operator * convection_z

**푸리에 변환:**

$$
\hat{\mathbf{m}} = \mathcal{F}[\mathbf{u} \times \boldsymbol{\omega}]
$$

##### 단계 5: De-aliasing 적용 (줄 328-331)


In [ ]:
convection_x_fft .*= dealias
convection_y_fft .*= dealias
convection_z_fft .*= dealias

**수학적 연산:**

$$
\hat{\mathbf{m}}_{filtered} = H(\mathbf{k}) \cdot \hat{\mathbf{m}}
$$

**효과:**
- 고주파수 노이즈 제거
- 에너지 pile-up 방지
- 장시간 안정성 확보


##### 단계 6: 의사-압력 (pseudo-pressure) 계산 (줄 333-340)


In [ ]:
pressure_fft = (
    normalized_wavenumbers_x .* convection_x_fft
    +
    normalized_wavenumbers_y .* convection_y_fft
    +
    normalized_wavenumbers_z .* convection_z_fft
)

**수학적 유도:**

1. **발산 조건**:
   
  $$
  \nabla \cdot \mathbf{u}^{n+1} = 0
  $$

2. **압력 푸아송 방정식**:
   
  $$
  \nabla^2 p = -\nabla \cdot (\mathbf{u} \times \boldsymbol{\omega})
  $$

3. **푸리에 공간 해**:
   
  $$
  \hat{p} = \frac{i\mathbf{k} \cdot \hat{\mathbf{m}}}{|\mathbf{k}|^2}
  $$

실제 구현에서는 정규화된 파수를 사용:

$$
\hat{p} = \hat{k}_x \cdot \hat{m}_x + \hat{k}_y \cdot \hat{m}_y + \hat{k}_z \cdot \hat{m}_z
$$

여기서 $\hat{k}_i = k_i / |\mathbf{k}|$ 는 정규화된 파수 성분

##### 단계 7: RHS 계산 (줄 343-363)

In [ ]:
rhs_x_fft = (
    convection_x_fft                                        # 대류항
    - KINEMATIC_VISCOSITY * wavenumbers_norm.^2 .* velocity_x_fft  # 점성 확산
    - normalized_wavenumbers_x .* pressure_fft              # 압력 구배
)

rhs_y_fft = (
    convection_y_fft
    - KINEMATIC_VISCOSITY * wavenumbers_norm.^2 .* velocity_y_fft
    - normalized_wavenumbers_y .* pressure_fft
)

rhs_z_fft = (
    convection_z_fft
    - KINEMATIC_VISCOSITY * wavenumbers_norm.^2 .* velocity_z_fft
    - normalized_wavenumbers_z .* pressure_fft
)

**각 항의 수학적 의미:**

1. **대류항**: 
   
   $$
   \mathbf{u} \times \boldsymbol{\omega}
   $$

2. **점성 확산**:
   
   $$
   -\nu |\mathbf{k}|^2 \hat{\mathbf{u}}
   $$ 
   
3. **압력 투영**: 
   
   $$
   -\hat{\mathbf{k}} \hat{p} / |\mathbf{k}|
   $$

**전체 RHS:**

$$
\frac{d\hat{\mathbf{u}}}{dt} = \hat{\mathbf{m}} - \nu |\mathbf{k}|^2 \hat{\mathbf{u}} - \frac{\hat{\mathbf{k}} \hat{p}}{|\mathbf{k}|}
$$

##### 단계 8: 시간 적분 (줄 365-368)

In [ ]:
velocity_x_fft .+= rhs_x_fft * TIME_STEP_LENGTH
velocity_y_fft .+= rhs_y_fft * TIME_STEP_LENGTH
velocity_z_fft .+= rhs_z_fft * TIME_STEP_LENGTH

**명시적 오일러 (Explicit Euler) 방법:**

$$
\hat{\mathbf{u}}^{n+1} = \hat{\mathbf{u}}^n + \Delta t \cdot \frac{d\hat{\mathbf{u}}}{dt}
$$

**시간 적분 오차:**
- 국소 절단 오차: $\mathcal{O}(\Delta t^2)$
- 전역 오차: $\mathcal{O}(\Delta t)$

##### 단계 9: 물리 공간으로 변환 (줄 370-373)


In [ ]:
velocity_x .= real(fft_operator \ velocity_x_fft)
velocity_y .= real(fft_operator \ velocity_y_fft)
velocity_z .= real(fft_operator \ velocity_z_fft)

**역 푸리에 변환:**

$$
\mathbf{u}^{n+1}(\mathbf{x}) = \mathcal{F}^{-1}[\hat{\mathbf{u}}^{n+1}(\mathbf{k})]
$$

### 시각화 (줄 376)

In [ ]:
if t % PLOT_EVERY == 0  # 3 단계마다

**시각화 주기:**
- 실제 시간 간격: 3 × 0.02 = 0.06 시간 단위
- 총 시각화 프레임: 1700 ÷ 3 ≈ 567 프레임

#### 와도 크기 계산 (줄 377-386)

In [ ]:
curl_x_squared .= curl_x.^2
curl_y_squared .= curl_y.^2
curl_z_squared .= curl_z.^2
curl_magnitude .= sqrt.(
    curl_x_squared + curl_y_squared + curl_z_squared
)

**와도 크기:**

$$
|\boldsymbol{\omega}| = \sqrt{\omega_x^2 + \omega_y^2 + \omega_z^2}
$$

**물리적 의미:**
- 국소 회전 강도
- 난류 구조 지표
- 에너지 소산과 직접 관련: $\epsilon = 2\nu \langle|\boldsymbol{\omega}|^2\rangle$

In [ ]:
delete!(makie_plot_3d_contour.parent, makie_plot_3d_contour)

#### 3D 등고선 플롯 (줄 390-398)

In [ ]:
makie_plot_3d_contour = contour!(
    makie_plot_axis,
    x_range, y_range, z_range,
    curl_magnitude,
    alpha = 0.01,              # 투명도 (거의 투명)
    levels = range(1.0, 10.0, length=5),  # 등고선 레벨
)

**등고선 레벨:**
- 범위: $|\boldsymbol{\omega}| \in [1, 10]$
- 5개 레벨: [1.0, 3.25, 5.5, 7.75, 10.0]
- 시간에 따라 동적 조정 가능

## 결과

```{raw} html
<div style="position: relative; padding-bottom: 56.25%; height: 0; overflow: hidden;">
  <iframe 
    src="https://www.youtube.com/embed/4uzWBS4Q100?si=SJgeMAoiApdnAFuf" 
    style="position: absolute; top: 0; left: 0; width: 100%; height: 100%;"
    frameborder="0" 
    allowfullscreen>
  </iframe>
</div>
```

### 물리적 진화 과정:

1. Laminar vortex sheets (t ≈ 0-10)
2. Vortex stretching (t ≈ 10-15)
3. Breakdown to turbulence (t ≈ 15-25)
4. Fully turbulent state (t ≈ 25-34)

## 결론

이 시뮬레이션은 Taylor-Green 와류가 난류로 발전하고 소멸하는 과정을 재현한다. 초기에 큰 규모의 소용돌이 구조가 존재하는데, 시간이 지날수록 이 소용돌이가 늘어나고(stretching) 서로 재연결(reconnection)되며, 더 작은 소용돌이가 차례대로 생긴다(energy cascade). 결국 점성에 의해 가장 작은 규모에서 에너지가 열로 소산(dissipation)되며, 전체 에너지가 감소한다. 
이 때의 에너지 스펙트럼은 Kolmogorov -5/3 법칙을 따른다: $E(k) \propto k^{-5/3}$

추가적으로, 이 시뮬레이션에서 에너지 보존과 소산 과정에 대해 직접 확인해보면 좋을 것 같다.

**운동 에너지 (kinetic energy)**

$$
E(t) = \frac{1}{2} \int_\Omega |\mathbf{u}|^2 d\mathbf{x} = \frac{1}{2} \sum_{\mathbf{k}} |\hat{\mathbf{u}}(\mathbf{k})|^2
$$

**엔스트로피 (enstrophy)**

$$
\Omega(t) = \frac{1}{2} \int_\Omega |\boldsymbol{\omega}|^2 d\mathbf{x} = \frac{1}{2} \sum_{\mathbf{k}} |\mathbf{k}|^2 |\hat{\mathbf{u}}(\mathbf{k})|^2
$$

**에너지 소산율 (energy dissipation rate)**

$$
\epsilon(t) = -\frac{dE}{dt} = \nu \int_\Omega |\boldsymbol{\omega}|^2 d\mathbf{x} = 2\nu \Omega(t)
$$



*P.S. Julia는 Jupyter Notebook에서 Animation 형태로 3D 시각화를 어떻게 하는지 모르겠다.. 찾아보자..*